# DeepExtractor: glitch reconstruction and validation tutorial with Virgo model

This notebook illustrates how to reconstruct real glitches from Virgo's O3 observing run using the DeepExtractor model tailored for Virgo data. We fetch open data from GWOSC, then whiten it, and call `model.reconstruct()` to obtain glitch estimates. Finally we call `model.background()` to retrieve the reconstructed background and validate it using `glitchfind`, a data quality tool that performs a p-value test to assess whether the reconstructed background contains some excess power or not.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import scienceplots
import importlib

from gwpy.timeseries import TimeSeries
from pycbc.types import TimeSeries as TimeSeries_pycbc

import deepextractor
from deepextractor.utils.checkpoints import CHECKPOINT_REAL
from deepextractor.utils.visualization import plot_q_transform
from deepextractor.utils.signal import custom_whiten

plt.style.use(['science'])
plt.rcParams['text.usetex'] = False

TimeSeries_pycbc.custom_whiten = custom_whiten

In [2]:
import importlib.resources as pkg_resources

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 4096

# Resolve bundled assets from the installed package
_assets = pkg_resources.files("deepextractor") / "assets"
ASSETS_DIR = str(_assets)

# Bundled GravitySpy O3a sample (10 high-SNR H1 examples per class)
# Source: GravitySpy LIGO O3a high-confidence dataset — https://doi.org/10.5281/zenodo.1476551
GRAVITY_SPY_CSV = str(_assets / "data_o3a_sample.csv")

## Gravity Spy dataset 

We use a bundled sample of the high-confidence GravitySpy O3a catalogue to identify specific glitch events by GPS time. This is a real subset of the [GravitySpy LIGO O3a dataset](https://zenodo.org/records/1476551), containing 10 high-SNR V1 examples per glitch class. 

Each row records the GPS time of a glitch trigger, its SNR, its GravitySpy label, **something else**. We pick one Blip and one Scattered Light event for reconstruction.